In [2]:
!pip install yfinance

# Data Collection

Historical stock market data for AAPL, MSFT, and SPY is downloaded using the yfinance library. The raw dataset is saved before any preprocessing to ensure reproducibility.

In [3]:
#Set up project & install yfinance, statsmodels, prophet
import yfinance as yf
import pandas as pd
import statsmodels as stats
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
yf.__version__

'0.2.66'

In [41]:
#Fetch 5-year OHLCV data for AAPL, MSFT, and SPY
data = yf.download(["AAPL", "MSFT", "SPY"], period="5y", auto_adjust=True)

[*********************100%***********************]  3 of 3 completed


In [42]:
data.to_csv("raw_stock_data.csv")


In [18]:
from google.colab import files
files.download("raw_stock_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Initial Data Inspection

The dataset is inspected to understand its structure, data types, summary statistics, missing values, and duplicate records before performing any cleaning operations.

In [43]:
data.head()

Price            Close                                High              \
Ticker            AAPL        MSFT         SPY        AAPL        MSFT   
Date                                                                     
2021-06-30  133.502106  259.981873  400.078125  133.940741  260.423325   
2021-07-01  133.804169  260.653656  402.293091  133.862651  260.883974   
2021-07-02  136.426270  266.459839  405.368011  136.465253  266.795739   
2021-07-06  138.434280  266.469421  404.629669  139.535740  268.110495   
2021-07-07  140.919876  268.647919  406.059662  141.231789  269.377298   

Price                          Low                                Open  \
Ticker             SPY        AAPL        MSFT         SPY        AAPL   
Date                                                                     
2021-06-30  400.751060  132.439614  258.734278  399.255645  132.732043   
2021-07-01  402.451991  132.332284  258.734262  400.769638  133.151087   
2021-07-02  405.723176  134.272061  261.517406  402.377182  134.418268   
2021-07-06  405.639086  136.533517  263.244825  401.900563  136.533517   
2021-07-07  406.340068  139.058097  265.979963  403.302518  139.915868   

Price                                  Volume                      
Ticker            MSFT         SPY       AAPL      MSFT       SPY  
Date                                                               
2021-06-30  259.780344  399.283683   63261400  21656500  64827900  
2021-07-01  258.743839  400.835069   52485800  16725300  53441000  
2021-07-02  261.824516  403.452030   78852600  26458000  57697700  
2021-07-06  266.824505  405.424111  108181800  31565600  68710400  
2021-07-07  268.139281  405.311969  104911600  23260000  63549500

In [44]:
data.shape

(1254, 15)

In [45]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1254 entries, 2021-06-30 to 2026-06-29
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   (Close, AAPL)   1254 non-null   float64
 1   (Close, MSFT)   1254 non-null   float64
 2   (Close, SPY)    1254 non-null   float64
 3   (High, AAPL)    1254 non-null   float64
 4   (High, MSFT)    1254 non-null   float64
 5   (High, SPY)     1254 non-null   float64
 6   (Low, AAPL)     1254 non-null   float64
 7   (Low, MSFT)     1254 non-null   float64
 8   (Low, SPY)      1254 non-null   float64
 9   (Open, AAPL)    1254 non-null   float64
 10  (Open, MSFT)    1254 non-null   float64
 11  (Open, SPY)     1254 non-null   float64
 12  (Volume, AAPL)  1254 non-null   int64  
 13  (Volume, MSFT)  1254 non-null   int64  
 14  (Volume, SPY)   1254 non-null   int64  
dtypes: float64(12), int64(3)
memory usage: 156.8 KB


In [46]:
data.describe()

Price         Close                                   High               \
Ticker         AAPL         MSFT          SPY         AAPL         MSFT   
count   1254.000000  1254.000000  1254.000000  1254.000000  1254.000000   
mean     193.593822   358.616016   499.210957   195.532880   361.975144   
std       44.716047    83.738895   111.511751    45.112576    84.089463   
min      122.933556   207.734009   339.378540   125.637661   213.706668   
25%      156.210091   282.758560   406.741707   158.730805   284.527334   
50%      182.407791   366.283310   458.634506   184.130751   368.843527   
75%      226.514889   418.293060   585.842545   228.467952   421.501002   
max      315.200012   538.658569   757.618225   317.399994   551.048474   

Price                        Low                                   Open  \
Ticker          SPY         AAPL         MSFT          SPY         AAPL   
count   1254.000000  1254.000000  1254.000000  1254.000000  1254.000000   
mean     501.864849   191.509875   355.024199   496.106292   193.417091   
std      111.637069    44.385245    83.408843   111.264467    44.754468   
min      342.481461   122.097746   206.938912   331.335655   123.907026   
25%      409.257258   154.276039   279.159455   402.919768   156.502153   
50%      459.464486   180.638115   362.538249   455.610718   182.044389   
75%      588.762484   224.232815   414.080919   582.950134   226.519102   
max      758.446109   309.649994   537.366763   754.805464   314.179993   

Price                                   Volume                              
Ticker         MSFT          SPY          AAPL          MSFT           SPY  
count   1254.000000  1254.000000  1.254000e+03  1.254000e+03  1.254000e+03  
mean     358.632730   499.100338  6.531308e+07  2.651506e+07  7.572433e+07  
std       83.868034   111.526423  2.874143e+07  1.203848e+07  2.909015e+07  
min      210.933620   332.382655  1.791060e+07  5.855900e+06  2.604870e+07  
25%      281.722584   406.585267  4.578078e+07  1.900120e+07  5.681018e+07  
50%      365.189541   457.765711  5.728620e+07  2.362185e+07  7.120585e+07  
75%      418.234886   586.716956  7.740915e+07  3.065028e+07  8.941912e+07  
max      550.830186   756.201867  3.186799e+08  1.862016e+08  2.566114e+08

In [47]:
data.isnull().sum().sum()

np.int64(0)

# Missing Values & Data Cleaning

The dataset is checked for missing values, duplicate records, and non-trading day gaps. Appropriate cleaning operations are applied only if required.


In [48]:
data.duplicated().sum()

np.int64(0)

In [50]:
data.dtypes

Price   Ticker
Close   AAPL      float64
        MSFT      float64
        SPY       float64
High    AAPL      float64
        MSFT      float64
        SPY       float64
Low     AAPL      float64
        MSFT      float64
        SPY       float64
Open    AAPL      float64
        MSFT      float64
        SPY       float64
Volume  AAPL        int64
        MSFT        int64
        SPY         int64
dtype: object

In [54]:
data = data.ffill()

In [77]:
log_returns = np.log(data["Close"] / data["Close"].shift(1)).dropna()
log_returns.head()

Ticker,AAPL,MSFT,SPY
Date,,,
2021-07-01,0.002260,0.002581,0.005521
2021-07-02,0.019407,0.022031,0.007614
2021-07-06,0.014611,0.000036,-0.001823
2021-07-07,0.017796,0.008142,0.003528
2021-07-08,-0.009242,-0.009007,-0.008181


In [78]:
data.columns

MultiIndex([( 'Close', 'AAPL'),
            ( 'Close', 'MSFT'),
            ( 'Close',  'SPY'),
            (  'High', 'AAPL'),
            (  'High', 'MSFT'),
            (  'High',  'SPY'),
            (   'Low', 'AAPL'),
            (   'Low', 'MSFT'),
            (   'Low',  'SPY'),
            (  'Open', 'AAPL'),
            (  'Open', 'MSFT'),
            (  'Open',  'SPY'),
            ('Volume', 'AAPL'),
            ('Volume', 'MSFT'),
            ('Volume',  'SPY')],
           names=['Price', 'Ticker'])

# Data Type Validation & Memory Optimization

The data types are verified and optimized to reduce memory usage while preserving data accuracy.

In [79]:
data = data.astype({
    col: "float32"
    for col in data.select_dtypes("float64").columns
})

data = data.astype({
    col: "int32"
    for col in data.select_dtypes("int64").columns
})

In [80]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1254 entries, 2021-06-30 to 2026-06-29
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   (Close, AAPL)   1254 non-null   float32
 1   (Close, MSFT)   1254 non-null   float32
 2   (Close, SPY)    1254 non-null   float32
 3   (High, AAPL)    1254 non-null   float32
 4   (High, MSFT)    1254 non-null   float32
 5   (High, SPY)     1254 non-null   float32
 6   (Low, AAPL)     1254 non-null   float32
 7   (Low, MSFT)     1254 non-null   float32
 8   (Low, SPY)      1254 non-null   float32
 9   (Open, AAPL)    1254 non-null   float32
 10  (Open, MSFT)    1254 non-null   float32
 11  (Open, SPY)     1254 non-null   float32
 12  (Volume, AAPL)  1254 non-null   int32  
 13  (Volume, MSFT)  1254 non-null   int32  
 14  (Volume, SPY)   1254 non-null   int32  
dtypes: float32(12), int32(3)
memory usage: 83.3 KB


In [81]:
data.memory_usage(deep=True)

,0
Index,10032
"(Close, AAPL)",5016
"(Close, MSFT)",5016
"(Close, SPY)",5016
"(High, AAPL)",5016
"(High, MSFT)",5016
"(High, SPY)",5016
"(Low, AAPL)",5016
"(Low, MSFT)",5016
"(Low, SPY)",5016


# Final Validation

The cleaned dataset is validated by checking data types, missing values, and duplicate records before exporting it for further analysis.

In [82]:
cleaned_data = data.copy()
cleaned_data.to_csv("cleaned_stock_data.csv")
files.download("cleaned_stock_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>